In [ ]:
"""
Knee abnormality report labeler — small model, batched, multi-GPU.

Same architecture as the original Qwen script (prompt -> JSON -> parse), but:
  * Qwen2.5-1.5B-Instruct instead of 7B  (~3GB fp16, fits trivially on one T4)
  * fp16, no quantization  (nf4 is SLOWER than fp16 on Turing; T4 has no bf16)
  * batched generation      (the actual 15-25x speedup)
  * chat template applied   (Instruct models degrade badly without it)
  * one full model replica per GPU, not device_map="auto" layer sharding

Kaggle 2x T4.
"""

import json
import logging
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import pydicom
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # step up to -3B- if parse quality is poor
MODEL_NAME = "/kaggle/input/datasets/soumabhamajumdar2548/qwen25-1b5-instruct"
COMP_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
BATCH_SIZE = 32          # drop to 16 if you see OOM with long reports
MAX_INPUT_TOKENS = 1536
MAX_NEW_TOKENS = 96      # a 12-key JSON is ~70 tokens; 256 was wasting decode steps
CHECKPOINT_EVERY = 5     # batches

LABEL_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

SYSTEM_PROMPT = (
    "You are a musculoskeletal radiologist extracting structured findings from "
    "knee MRI reports. You reply with a single JSON object and nothing else."
)


# --------------------------------------------------------------------------- #
# DICOM metadata (unchanged behaviour, trimmed fields)
# --------------------------------------------------------------------------- #

class DicomMeta:
    KEEP = ("Modality", "SeriesDescription", "BodyPartExamined", "Laterality")

    def __init__(self, base_path):
        self.base_path = base_path
        self.cache = {}

    def get(self, study_uid):
        if study_uid in self.cache:
            return self.cache[study_uid]

        study_path = Path(self.base_path) / str(study_uid)
        if not study_path.exists():
            self.cache[study_uid] = {}
            return {}

        dcm_files = list(study_path.rglob("*.dcm"))
        if not dcm_files:
            self.cache[study_uid] = {}
            return {}

        try:
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            meta = {}
            for key in self.KEEP:
                val = ds.get(key, "")
                val = "" if val is None else str(val).strip()
                if val and val.upper() not in ("N/A", "NONE", "NULL"):
                    meta[key] = val
            self.cache[study_uid] = meta
            return meta
        except Exception as e:  # noqa: BLE001
            logging.warning(f"Error reading DICOM for {study_uid}: {e}")
            self.cache[study_uid] = {}
            return {}


# --------------------------------------------------------------------------- #
# One model replica pinned to one GPU
# --------------------------------------------------------------------------- #

class Replica:
    def __init__(self, model_name, device):
        self.device = device
        logging.info(f"Loading {model_name} onto {device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).to(device)
        self.model.eval()
        self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
        logging.info(f"Ready on {device}")

    @torch.inference_mode()
    def generate(self, prompts):
        texts = [
            self.tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in prompts
        ]
        enc = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(self.device)

        out = self.model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        gen = out[:, enc["input_ids"].shape[1]:]
        return self.tokenizer.batch_decode(gen, skip_special_tokens=True)


# --------------------------------------------------------------------------- #
# Labeler
# --------------------------------------------------------------------------- #

class KneeAbnormalityLabeler:
    def __init__(self, model_name=MODEL_NAME,
                 checkpoint_path="labeling_checkpoint.json",
                 dicom_base_path=f"{COMP_ROOT}/train_series/",
                 n_gpus=None):
        self.checkpoint_path = checkpoint_path
        self.dicom = DicomMeta(dicom_base_path)
        self.results = {}

        if os.path.exists(checkpoint_path):
            with open(checkpoint_path) as f:
                self.results = json.load(f)
            logging.info(f"Loaded checkpoint with {len(self.results)} completed rows")

        if n_gpus is None:
            n_gpus = max(1, torch.cuda.device_count())
        devices = [f"cuda:{i}" for i in range(n_gpus)] if torch.cuda.is_available() else ["cpu"]
        self.replicas = [Replica(model_name, d) for d in devices]
        self.pool = ThreadPoolExecutor(max_workers=len(self.replicas))
        logging.info(f"{len(self.replicas)} replica(s) live")

    # ---- prompt ---------------------------------------------------------- #

    def create_prompt(self, report_text, dicom_metadata=None):
        keys = ", ".join(f'"{c}"' for c in LABEL_COLUMNS)
        prompt = (
            "Read the knee MRI report below and decide, for each of 12 conditions, "
            "whether it is present.\n\n"
            f"Output a JSON object with exactly these keys: {keys}\n\n"
            "Values:\n"
            "  1  = condition is affirmatively described\n"
            "  0  = condition is explicitly negated or described as normal/intact\n"
            '  "?" = not mentioned, or the report is equivocal\n\n'
            "Do not infer a condition from an adjacent one. A meniscal tear does not "
            "imply an ACL tear. Respect laterality: medial findings go to medial keys "
            "only.\n\n"
        )
        if dicom_metadata:
            prompt += "Acquisition details:\n"
            for k, v in dicom_metadata.items():
                prompt += f"- {k}: {v}\n"
            prompt += "\n"
        prompt += f"REPORT:\n{report_text}\n\nJSON:"
        return prompt

    # ---- parsing (kept from original, slightly tightened) ---------------- #

    @staticmethod
    def parse_response(response_text):
        if not response_text:
            return None

        cleaned = re.sub(r"```(?:json)?\s*", "", response_text, flags=re.IGNORECASE)
        cleaned = re.sub(r"```\s*$", "", cleaned)

        start = cleaned.find("{")
        if start == -1:
            return None
        end = cleaned.rfind("}")

        if end != -1 and start < end:
            try:
                return json.loads(cleaned[start:end + 1])
            except json.JSONDecodeError:
                pass

        # Truncated output: close the object and drop any trailing partial pair.
        tail = cleaned[start:]
        tail = re.sub(r",\s*\"[^\"]*\"?\s*:?\s*[^,}]*$", "", tail)
        try:
            return json.loads(tail + "}")
        except json.JSONDecodeError:
            pass

        match = re.search(r"\{[^{}]*\}", cleaned)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return None

    # ---- batch dispatch --------------------------------------------------- #

    def _run_batch(self, prompts):
        """Split one batch across replicas, run in parallel, reassemble in order."""
        n = len(self.replicas)
        if n == 1:
            return self.replicas[0].generate(prompts)

        chunks = [prompts[i::n] for i in range(n)]
        futures = [
            self.pool.submit(rep.generate, chunk)
            for rep, chunk in zip(self.replicas, chunks) if chunk
        ]
        outs = [f.result() for f in futures]

        merged = [None] * len(prompts)
        for i, out in enumerate(outs):
            merged[i::n] = out
        return merged

    def save_checkpoint(self):
        with open(self.checkpoint_path, "w") as f:
            json.dump(self.results, f)
        logging.info(f"Checkpoint saved: {len(self.results)} rows")

    # ---- main loop -------------------------------------------------------- #

    def label_dataframe(self, df):
        result_df = df.copy()
        if "inferred" not in result_df.columns:
            result_df["inferred"] = ""

        todo = []
        for idx in range(len(df)):
            if str(idx) in self.results:
                continue
            row = df.iloc[idx]
            if not any(pd.isna(row[c]) for c in LABEL_COLUMNS):
                continue
            if pd.isna(row["Report"]) or not str(row["Report"]).strip():
                self.results[str(idx)] = {"error": "empty_report"}
                continue
            todo.append(idx)

        logging.info(f"{len(todo)} rows need labeling")

        n_batches = (len(todo) + BATCH_SIZE - 1) // BATCH_SIZE
        parse_failures = 0

        for b in tqdm(range(n_batches), desc="Labeling"):
            batch_idx = todo[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            prompts = []
            for idx in batch_idx:
                row = df.iloc[idx]
                meta = self.dicom.get(row["StudyInstanceUID"])
                prompts.append(self.create_prompt(str(row["Report"]), meta))

            responses = self._run_batch(prompts)

            for idx, resp in zip(batch_idx, responses):
                labels = self.parse_response(resp)
                if labels is None:
                    parse_failures += 1
                    self.results[str(idx)] = {"error": "parse_failed", "raw": resp[:200]}
                    continue

                inferred_cols = []
                for col in LABEL_COLUMNS:
                    if col not in labels:
                        continue
                    val = labels[col]
                    if val == "?" or val is None:
                        continue
                    try:
                        num = float(val)
                    except (ValueError, TypeError):
                        continue
                    if num in (0.0, 1.0):
                        result_df.at[idx, col] = int(num)
                        inferred_cols.append(col)

                if inferred_cols:
                    result_df.at[idx, "inferred"] = ",".join(inferred_cols)
                self.results[str(idx)] = {"completed": True, "inferred": inferred_cols}

            if (b + 1) % CHECKPOINT_EVERY == 0:
                self.save_checkpoint()

        self.save_checkpoint()
        logging.info(f"Parse failures: {parse_failures} / {len(todo)}")
        return result_df


def main():
    df = pd.read_csv(f"{COMP_ROOT}/train.csv")
    logging.info(f"Loaded {len(df)} rows; columns: {df.columns.tolist()}")

    labeler = KneeAbnormalityLabeler()
    labeled_df = labeler.label_dataframe(df)
    labeled_df.to_csv("train_labeled.csv", index=False)

    print("\n" + "=" * 50)
    print("LABELING SUMMARY")
    print("=" * 50)
    print(f"Total rows: {len(labeled_df)}")
    print(f"Rows with inferred labels: {labeled_df['inferred'].str.len().gt(0).sum()}")
    print("\nLabel distribution:")
    for col in LABEL_COLUMNS:
        non_nan = labeled_df[col].notna().sum()
        inferred = labeled_df["inferred"].str.contains(re.escape(col), na=False).sum()
        print(f"  {col:20s}: {non_nan:6d} total, {inferred:6d} inferred")


if __name__ == "__main__":
    main()

In [ ]:
"""
Phase A, step 1 — build the slice manifest.

Walks the competition DICOM tree and produces one row per slice. No pixels are
read here; this reads DICOM headers only, and everything downstream in the
preprocessing pipeline reads its inputs from the table this produces.

What each group of columns is for:

  identity     study_uid, series_uid, path
               how we group slices into series and series into studies.

  geometry     ipp_*, iop_*, depth, plane
               ipp / iop are the raw DICOM fields. `depth` is how far along the
               stacking direction each slice sits, and is what step 3 sorts by.
               `plane` (sagittal / coronal / axial) is computed from the
               geometry rather than parsed out of the free-text series
               description, which differs between hospitals.

  acquisition  series_description, laterality, series_number, instance_number
               `laterality` drives the left-knee flip in step 6.

  decoding     rescale_slope, rescale_intercept, photometric_interpretation
               needed by step 4. Grabbed now so step 4 never has to reopen
               headers.

  shape        rows, cols, pixel_spacing_*, slice_thickness, spacing_between
               used for sanity checks and for the resize decision in step 7.

Usage
-----
    from build_manifest import build_manifest, summarise_manifest

    df = build_manifest("train", limit=20)     # smoke test on 20 studies
    summarise_manifest(df)

    df = build_manifest("train")               # the real run
    df.to_parquet("/kaggle/working/manifest_train.parquet")
"""

from __future__ import annotations

import os
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from tqdm.auto import tqdm

COMP_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUTPUT_DIR = Path("/kaggle/working")

# Listing the fields explicitly lets pydicom skip the rest of the header. That
# matters when you are opening a few hundred thousand files.
DICOM_TAGS = [
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SeriesNumber",
    "InstanceNumber",
    "ImagePositionPatient",
    "ImageOrientationPatient",
    "SeriesDescription",
    "Laterality",
    "BodyPartExamined",
    "Modality",
    "Rows",
    "Columns",
    "PixelSpacing",
    "SliceThickness",
    "SpacingBetweenSlices",
    "PhotometricInterpretation",
    "RescaleSlope",
    "RescaleIntercept",
]

PLANES = ("sagittal", "coronal", "axial")


# --------------------------------------------------------------------------- #
# Geometry
#
# ImageOrientationPatient is six numbers: the first three say which way you
# travel through the patient as you step one pixel RIGHT in the image, the last
# three as you step one pixel DOWN. ImagePositionPatient is where the slice's
# top-left corner sits, in millimetres.
# --------------------------------------------------------------------------- #

def slice_normal(iop):
    """The direction the slices stack along — perpendicular to the image."""
    row_dir = np.asarray(iop[:3], dtype=float)
    col_dir = np.asarray(iop[3:], dtype=float)
    return np.cross(row_dir, col_dir)


def slice_depth(ipp, normal):
    """How far along the stacking direction this slice sits, in millimetres.

    One number per slice. Sorting a series by this puts it in anatomical
    order, which sorting by filename or InstanceNumber does not reliably do.
    """
    return float(np.dot(np.asarray(ipp, dtype=float), normal))


def plane_from_normal(normal):
    """sagittal / coronal / axial, from which patient axis the stack runs along.

    Patient axes are x = toward the left, y = toward the back, z = toward the
    head. Slices stacking along x are side views (sagittal), along y are front
    views (coronal), along z are top-down (axial).
    """
    return PLANES[int(np.argmax(np.abs(normal)))]


# --------------------------------------------------------------------------- #
# Reading one slice's header
# --------------------------------------------------------------------------- #

def _get(ds, name, default=None):
    value = getattr(ds, name, None)
    return default if value is None else value


def _floats(value):
    """DICOM multi-valued numbers come back as a MultiValue, not a list."""
    return None if value is None else [float(v) for v in value]


def read_slice_header(path: Path, study_dir_name: str) -> dict:
    ds = pydicom.dcmread(str(path), stop_before_pixels=True, specific_tags=DICOM_TAGS)

    ipp = _floats(_get(ds, "ImagePositionPatient"))
    iop = _floats(_get(ds, "ImageOrientationPatient"))
    spacing = _floats(_get(ds, "PixelSpacing"))

    row = {
        "study_uid": str(_get(ds, "StudyInstanceUID", "")),
        "study_dir": study_dir_name,
        "series_uid": str(_get(ds, "SeriesInstanceUID", "")),
        "path": str(path),

        "series_number": _get(ds, "SeriesNumber"),
        "instance_number": _get(ds, "InstanceNumber"),
        "series_description": str(_get(ds, "SeriesDescription", "")),
        "laterality": str(_get(ds, "Laterality", "")),
        "body_part": str(_get(ds, "BodyPartExamined", "")),
        "modality": str(_get(ds, "Modality", "")),

        "rows": _get(ds, "Rows"),
        "cols": _get(ds, "Columns"),
        "pixel_spacing_row": spacing[0] if spacing else None,
        "pixel_spacing_col": spacing[1] if spacing else None,
        "slice_thickness": _get(ds, "SliceThickness"),
        "spacing_between_slices": _get(ds, "SpacingBetweenSlices"),

        # Everything step 4 needs, so it never has to reopen the header.
        "photometric_interpretation": str(_get(ds, "PhotometricInterpretation", "")),
        "rescale_slope": _get(ds, "RescaleSlope"),
        "rescale_intercept": _get(ds, "RescaleIntercept"),

        "error": None,
    }

    # Raw geometry, kept so we never have to rescan.
    for i, key in enumerate(("ipp_x", "ipp_y", "ipp_z")):
        row[key] = ipp[i] if ipp else None
    for i, key in enumerate(("iop_rx", "iop_ry", "iop_rz", "iop_cx", "iop_cy", "iop_cz")):
        row[key] = iop[i] if iop else None

    # Derived geometry. Both come from the two vectors above, so we compute
    # them here rather than carrying nine columns around and redoing it later.
    if iop is not None:
        normal = slice_normal(iop)
        row["plane"] = plane_from_normal(normal)
        row["depth"] = slice_depth(ipp, normal) if ipp is not None else None
    else:
        row["plane"] = None
        row["depth"] = None

    return row


def scan_study(study_dir: Path) -> list[dict]:
    """Every .dcm under one study folder. Errors are recorded, not raised —
    one unreadable file should not kill a 40-minute scan."""
    rows = []
    for path in sorted(Path(study_dir).rglob("*.dcm")):
        try:
            rows.append(read_slice_header(path, Path(study_dir).name))
        except Exception as exc:  # noqa: BLE001
            rows.append({
                "path": str(path),
                "study_dir": Path(study_dir).name,
                "error": repr(exc),
            })
    return rows


# --------------------------------------------------------------------------- #
# Driver
# --------------------------------------------------------------------------- #

def build_manifest(split: str = "train", limit: int | None = None,
                   workers: int | None = None) -> pd.DataFrame:
    """One row per slice for every study in {split}_series/.

    `limit` caps the number of studies — use it to smoke-test on 20 studies
    before committing to the full scan.
    """
    root = COMP_ROOT / f"{split}_series"
    if not root.exists():
        raise FileNotFoundError(f"{root} does not exist")

    studies = sorted(p for p in root.iterdir() if p.is_dir())
    if limit is not None:
        studies = studies[:limit]

    workers = workers or os.cpu_count() or 2

    rows: list[dict] = []
    with ProcessPoolExecutor(max_workers=workers) as pool:
        for study_rows in tqdm(pool.map(scan_study, studies, chunksize=4),
                               total=len(studies), desc=f"scanning {split}"):
            rows.extend(study_rows)

    df = pd.DataFrame(rows)
    df["split"] = split
    return df


# --------------------------------------------------------------------------- #
# Summary
#
# This is the point of running the scan first: it answers the questions we
# would otherwise be guessing at for the rest of the pipeline.
# --------------------------------------------------------------------------- #

def summarise_manifest(df: pd.DataFrame) -> None:
    ok = df[df["error"].isna()] if "error" in df else df
    failed = len(df) - len(ok)

    print(f"slices        {len(df):,}   ({failed:,} failed to read)")
    print(f"series        {ok['series_uid'].nunique():,}")
    print(f"studies       {ok['study_uid'].nunique():,}")

    mismatch = (ok["study_uid"] != ok["study_dir"]).sum()
    print(f"folder name disagrees with header StudyInstanceUID: {mismatch:,} slices")

    print("\nplane")
    print(ok["plane"].value_counts(dropna=False).to_string())

    print("\nlaterality")
    print(ok["laterality"].value_counts(dropna=False).to_string())

    per_series = ok.groupby("series_uid").size()
    print("\nslices per series")
    print(per_series.describe().to_string())
    print(f"series with < 8 slices (step 2 drops these): "
          f"{(per_series < 8).sum():,} of {len(per_series):,}")

    # Answers 'do I actually need to apply rescale slope/intercept?'
    print("\n(rescale_slope, rescale_intercept) combinations")
    pairs = ok[["rescale_slope", "rescale_intercept"]].astype(str)
    print(pairs.value_counts(dropna=False).head(10).to_string())

    # Answers 'do I need the MONOCHROME1 inversion?'
    print("\nphotometric interpretation")
    print(ok["photometric_interpretation"].value_counts(dropna=False).to_string())

    print("\nimage size")
    print(ok[["rows", "cols"]].astype(str).value_counts().head(10).to_string())

    print("\nmost common series descriptions")
    print(ok["series_description"].value_counts().head(20).to_string())


# if __name__ == "__main__":
#     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     for split in ("train", "test"):
#         manifest = build_manifest(split)
#         out = OUTPUT_DIR / f"manifest_{split}.parquet"
#         manifest.to_parquet(out, index=False)
#         print(f"\n=== {split} -> {out} ===")
#         summarise_manifest(manifest)

manifest_train = build_manifest("train")
manifest_train.to_parquet("/kaggle/working/manifest_train.parquet", index=False)
summarise_manifest(manifest_train)

manifest_test = build_manifest("test")
manifest_test.to_parquet("/kaggle/working/manifest_test.parquet", index=False)
summarise_manifest(manifest_test)


scanning train:   0%|          | 0/4407 [00:00<?, ?it/s]

In [ ]:
"""
Phase A, step 2 — filter the manifest down to usable series.

Takes the slice manifest from step 1 and removes what we cannot or should not
train on. Nothing here reads pixels; it is all decisions made from the header
table.

Two kinds of removal happen, and it is worth keeping them straight:

  slice-level   individual slices that are unusable (failed to read, missing
                geometry, duplicate position). If enough of these go, the
                series may then fail a series-level rule.

  series-level  whole series we do not want (too short, localizers, wrong
                modality, inconsistent image size).

Every rejection is recorded with a reason rather than silently dropped, so you
can look at what went and decide whether a rule is too aggressive.

Usage
-----
    from filter_series import filter_manifest, summarise_filter

    kept, report = filter_manifest(manifest)
    summarise_filter(manifest, kept, report)

    # what got thrown away and why
    report[report.reject_reason.notna()].head(50)
"""

from __future__ import annotations

import re

import numpy as np
import pandas as pd

MIN_SLICES = 8

# Long distinctive words can be matched anywhere in the description; short ones
# need word boundaries so "loc" does not match some unrelated substring.
LOCALIZER_SUBSTRINGS = ("localizer", "localiser", "scout", "survey", "topogram",
                        "scanogram", "tracker", "smartbrain", "calibration",
                        "3-plane", "3 plane")
LOCALIZER_WORDS = re.compile(r"\b(loc|plan|ref|cal)\b", re.IGNORECASE)

# Slice positions are floats; round before comparing so two slices genuinely at
# the same place compare equal. 0.01 mm is far below any real slice gap.
DEPTH_ROUNDING = 2


def _looks_like_localizer(description: str) -> bool:
    text = (description or "").lower()
    if any(word in text for word in LOCALIZER_SUBSTRINGS):
        return True
    return bool(LOCALIZER_WORDS.search(text))


# --------------------------------------------------------------------------- #
# Slice-level cleaning
# --------------------------------------------------------------------------- #

def _drop_unusable_slices(manifest: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Remove slices we cannot place in space, and duplicate positions.

    Duplicate positions are the interesting case. Some series arrive with two
    slices at the same physical location — multi-echo sequences, or magnitude
    and phase images filed into one folder. Sorting those by depth produces a
    stack that alternates between two different images, which would poison the
    2.5D channel stack. We keep the first by instance number and flag it.
    """
    counts = {"start": len(manifest)}
    m = manifest

    if "error" in m.columns:
        m = m[m["error"].isna()]
    counts["read_errors"] = counts["start"] - len(m)

    before = len(m)
    m = m.dropna(subset=["depth", "plane", "series_uid"])
    counts["missing_geometry"] = before - len(m)

    before = len(m)
    m = m.assign(_depth_key=m["depth"].round(DEPTH_ROUNDING))
    m = (m.sort_values(["series_uid", "_depth_key", "instance_number"])
           .drop_duplicates(subset=["series_uid", "_depth_key"], keep="first")
           .drop(columns="_depth_key"))
    counts["duplicate_positions"] = before - len(m)

    return m, counts


# --------------------------------------------------------------------------- #
# Series-level summary and rules
# --------------------------------------------------------------------------- #

def build_series_table(manifest: pd.DataFrame) -> pd.DataFrame:
    """One row per series, with the aggregates the filter rules need."""
    m = manifest.sort_values(["series_uid", "depth"]).copy()

    # Gap between consecutive slices. In a healthy series these are all about
    # the same — that is the slice spacing. Wildly uneven gaps mean the folder
    # holds more than one acquisition.
    m["_gap"] = m.groupby("series_uid")["depth"].diff()
    gaps = m.groupby("series_uid")["_gap"]
    gap_mean = gaps.mean()
    gap_std = gaps.std()

    table = pd.DataFrame({
        "study_uid": m.groupby("series_uid")["study_uid"].first(),
        "n_slices": m.groupby("series_uid").size(),
        "plane": m.groupby("series_uid")["plane"].agg(lambda s: s.mode().iat[0]),
        "n_planes": m.groupby("series_uid")["plane"].nunique(),
        "series_description": m.groupby("series_uid")["series_description"].first(),
        "laterality": m.groupby("series_uid")["laterality"].first(),
        "modality": m.groupby("series_uid")["modality"].first(),
        "n_sizes": m.groupby("series_uid").apply(
            lambda g: g[["rows", "cols"]].drop_duplicates().shape[0],
            include_groups=False),
        "spacing_median": gaps.median(),
        "spacing_cv": (gap_std / gap_mean.abs().replace(0, np.nan)).abs(),
    })
    return table.reset_index()


def _reject_reason(row, min_slices: int) -> str | None:
    """First rule that fires wins, so the report reads as one reason per series.

    Ordered cheapest-and-most-certain first.
    """
    if str(row["modality"]).upper() not in ("MR", ""):
        return f"modality={row['modality']}"

    if _looks_like_localizer(row["series_description"]):
        return "localizer/scout"

    if row["n_slices"] < min_slices:
        return f"too few slices ({int(row['n_slices'])} < {min_slices})"

    # Different image dimensions inside one series cannot be stacked into a
    # single volume in step 7.
    if row["n_sizes"] > 1:
        return "inconsistent image size"

    # Slices facing different directions in one series means two acquisitions
    # were filed together.
    if row["n_planes"] > 1:
        return "mixed planes in one series"

    return None


def filter_manifest(manifest: pd.DataFrame, min_slices: int = MIN_SLICES,
                    ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns (kept slice manifest, per-series report).

    The report has a row for every series seen, with `reject_reason` set on the
    ones that were dropped and null on the ones that survived. Keeping the
    rejected rows is deliberate — a filter you cannot inspect is a filter you
    cannot trust.
    """
    cleaned, slice_counts = _drop_unusable_slices(manifest)

    report = build_series_table(cleaned)
    report["reject_reason"] = report.apply(_reject_reason, axis=1,
                                           min_slices=min_slices)

    keep_uids = set(report.loc[report["reject_reason"].isna(), "series_uid"])
    kept = cleaned[cleaned["series_uid"].isin(keep_uids)].copy()

    kept.attrs["slice_counts"] = slice_counts
    return kept, report


# --------------------------------------------------------------------------- #
# Summary
# --------------------------------------------------------------------------- #

def summarise_filter(original: pd.DataFrame, kept: pd.DataFrame,
                     report: pd.DataFrame) -> None:
    counts = kept.attrs.get("slice_counts", {})

    print("slice-level removals")
    for name in ("read_errors", "missing_geometry", "duplicate_positions"):
        print(f"  {name:22} {counts.get(name, 0):,}")

    print("\nseries-level removals")
    reasons = report["reject_reason"].fillna("KEPT")
    print(reasons.value_counts().to_string())

    print(f"\nslices   {len(original):,} -> {len(kept):,}")
    print(f"series   {report['series_uid'].nunique():,} -> "
          f"{kept['series_uid'].nunique():,}")
    print(f"studies  {original['study_uid'].nunique():,} -> "
          f"{kept['study_uid'].nunique():,}")

    # A study with nothing left cannot be predicted on. For train that is a
    # dropped row; for test it means you still owe a submission for it.
    lost = set(original["study_uid"].dropna()) - set(kept["study_uid"])
    print(f"\nstudies with no surviving series: {len(lost):,}")
    if lost:
        print("  " + ", ".join(sorted(lost)[:5]) + (" ..." if len(lost) > 5 else ""))

    print("\ndescriptions dropped as localizer/scout (check these are junk)")
    dropped = report[report["reject_reason"] == "localizer/scout"]
    print(dropped["series_description"].value_counts().head(15).to_string())

    kept_series = report[report["reject_reason"].isna()]
    print("\nsurviving series per study")
    print(kept_series.groupby("study_uid").size().describe().to_string())

    print("\nsurviving series by plane")
    print(kept_series["plane"].value_counts().to_string())

    # Not a rejection rule — uneven spacing is suspicious but not always fatal,
    # so it is reported for you to look at rather than acted on automatically.
    uneven = kept_series[kept_series["spacing_cv"] > 0.1]
    print(f"\nkept series with uneven slice spacing (cv > 0.1): {len(uneven):,}")

    missing_lat = kept_series["laterality"].isin(["", "nan", "None"]).sum()
    print(f"kept series with no Laterality field: {missing_lat:,} "
          f"(these need the geometry fallback in step 6)")


kept, report = filter_manifest(manifest_train)
summarise_filter(manifest_train, kept, report)     # three arguments